In [2]:
from os import chdir
from pathlib import Path
import time
cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import numpy as np
import itertools

In [4]:
# ------------------------------
# Matrix Factorization Model
# ------------------------------
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

        nn.init.normal_(self.user_factors.weight, std=0.1)
        nn.init.normal_(self.item_factors.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        dot = (self.user_factors(users) * self.item_factors(items)).sum(dim=1)
        return dot + self.user_bias(users).squeeze() + self.item_bias(items).squeeze() + self.global_bias



In [5]:
# ------------------------------
# Dataset Class
# ------------------------------
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_id"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


In [6]:
# ------------------------------
# Training and Evaluation
# ------------------------------
def train_model(model, train_loader, optimizer, criterion, device, known_users=None, known_items=None):
    model.train()
    total_loss = 0
    total_count = 0

    for users, items, ratings in train_loader:
        users = users.to(device)
        items = items.to(device)
        ratings = ratings.to(device)

        # Optional skip for unseen users/items
        if known_users is not None and not torch.all(torch.isin(users, known_users)):
            continue
        if known_items is not None and not torch.all(torch.isin(items, known_items)):
            continue

        optimizer.zero_grad()
        preds = model(users, items)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * ratings.size(0)
        total_count += ratings.size(0)

    if total_count == 0:
        return float("nan")

    return np.sqrt(total_loss / total_count)
    
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for users, items, ratings in loader:
            users, items, ratings = users.to(device), items.to(device), ratings.to(device)
            preds = model(users, items)
            loss = criterion(preds, ratings)
            total_loss += loss.item() * len(ratings)
    return np.sqrt(total_loss / len(loader.dataset))


In [7]:
train_df = pd.read_csv("dataset/ml100k_train_seed10.csv")
test_df = pd.read_csv("dataset/ml100k_test_seed10.csv")


n_users = max(train_df["user_id"].max(), test_df["user_id"].max()) + 1
n_items = max(train_df["item_id"].max(), test_df["item_id"].max()) + 1


print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")

Total User: 943
Total Item: 1633


In [8]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.MSELoss()

param_grid = {
    "n_factors": [10,16, 32, 64],
    "lr": [1e-3, 1e-2, 1e-1],
    "weight_decay": [0, 1e-4,1e-5]
}

best_val_rmse = float("inf")
best_params = {}
best_model_state = None
total_comm_cost = 0
total_grad_shared = 0
total_message_count = 0

for n_factors, lr, weight_decay in itertools.product(
    param_grid["n_factors"], param_grid["lr"], param_grid["weight_decay"]
):
    print(f"Testing n_factors={n_factors}, lr={lr}, weight_decay={weight_decay}")
    model = MatrixFactorization(n_users, n_items, n_factors).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
    val_loader = DataLoader(RatingsDataset(val_df), batch_size=1024)

    known_users_all = torch.unique(torch.tensor(train_df["user_id"].values)).to(device)
    known_items_all = torch.unique(torch.tensor(train_df["item_id"].values)).to(device)

    comm_cost_epoch = 0
    grad_share_epoch = 0
    message_count_epoch = 0
    epoch_num = 150
    num_sample = 800  # Number of users you want to sample

    for epoch in range( epoch_num):  # tuning loop
        
        if num_sample <= len(known_users_all):
            known_users = known_users_all[torch.randperm(len(known_users_all))[:num_sample]]
        else:
            raise ValueError("Requested sample size exceeds number of known users.")

        known_items = known_items_all
        
        train_rmse = train_model(model, train_loader, optimizer, criterion, device, known_users, known_items)

        if epoch == 0:
            param_bytes = sum(p.numel() for p in model.parameters()) * 4
            comm_cost_epoch = param_bytes
            grad_share_epoch = sum(p.numel() for p in model.parameters())
            message_count_epoch = 1  # assume 1 message per epoch

        total_comm_cost += comm_cost_epoch 
        total_grad_shared += grad_share_epoch 
        total_message_count += message_count_epoch 
    
    val_rmse = evaluate_model(model, val_loader, criterion, device)
    print(f"  => Val RMSE: {val_rmse:.4f}")
    
    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_params = {
            "n_factors": n_factors,
            "lr": lr,
            "weight_decay": weight_decay
        }
        best_model_state = model.state_dict()

Testing n_factors=10, lr=0.001, weight_decay=0
  => Val RMSE: 3.7024
Testing n_factors=10, lr=0.001, weight_decay=0.0001
  => Val RMSE: 3.7026
Testing n_factors=10, lr=0.001, weight_decay=1e-05
  => Val RMSE: 3.7027
Testing n_factors=10, lr=0.01, weight_decay=0
  => Val RMSE: 3.7022
Testing n_factors=10, lr=0.01, weight_decay=0.0001
  => Val RMSE: 3.7023
Testing n_factors=10, lr=0.01, weight_decay=1e-05
  => Val RMSE: 3.7026
Testing n_factors=10, lr=0.1, weight_decay=0
  => Val RMSE: 3.7025
Testing n_factors=10, lr=0.1, weight_decay=0.0001
  => Val RMSE: 3.7027
Testing n_factors=10, lr=0.1, weight_decay=1e-05
  => Val RMSE: 3.7025
Testing n_factors=16, lr=0.001, weight_decay=0
  => Val RMSE: 3.7030
Testing n_factors=16, lr=0.001, weight_decay=0.0001
  => Val RMSE: 3.7023
Testing n_factors=16, lr=0.001, weight_decay=1e-05
  => Val RMSE: 3.7027
Testing n_factors=16, lr=0.01, weight_decay=0
  => Val RMSE: 3.7030
Testing n_factors=16, lr=0.01, weight_decay=0.0001
  => Val RMSE: 3.7029
Test

In [9]:
print("\nBest Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
    print(f"✅ Best Validation RMSE: {best_val_rmse:.4f}")
    print(f"📦 Total Communication Cost: {total_comm_cost / 1e6:.2f} MB")
    print(f"🔁 Total Gradients Shared: {total_grad_shared}")
    print(f"📨 Total Messages Sent: {total_message_count}")



Best Parameters:
  n_factors: 32
✅ Best Validation RMSE: 3.7019
📦 Total Communication Cost: 1752.73 MB
🔁 Total Gradients Shared: 438183000
📨 Total Messages Sent: 5400
  lr: 0.001
✅ Best Validation RMSE: 3.7019
📦 Total Communication Cost: 1752.73 MB
🔁 Total Gradients Shared: 438183000
📨 Total Messages Sent: 5400
  weight_decay: 0.0001
✅ Best Validation RMSE: 3.7019
📦 Total Communication Cost: 1752.73 MB
🔁 Total Gradients Shared: 438183000
📨 Total Messages Sent: 5400


In [10]:
test_loader = DataLoader(RatingsDataset(test_df), batch_size=1024)
final_model = MatrixFactorization(n_users, n_items, best_params["n_factors"]).to(device)
final_model.load_state_dict(best_model_state)
test_rmse = evaluate_model(final_model, test_loader, criterion, device)
print(f"✅ Test RMSE with best model: {test_rmse:.4f}")


✅ Test RMSE with best model: 3.7062


In [11]:
best_params

{'n_factors': 32, 'lr': 0.001, 'weight_decay': 0.0001}

In [12]:
total_comm_cost = 0
total_grad_shared = 0
total_message_count = 0

comm_cost_log = []
grad_share_log = []
message_count_log = []
epoch_time_log = []
rmse_log = []
model = MatrixFactorization(n_users, n_items, best_params["n_factors"]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"], weight_decay=best_params['weight_decay'])

train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
val_loader = DataLoader(RatingsDataset(val_df), batch_size=1024)

known_users = torch.unique(torch.tensor(train_df["user_id"].values)).to(device)
known_items = torch.unique(torch.tensor(train_df["item_id"].values)).to(device)

comm_cost_epoch = 0
grad_share_epoch = 0
message_count_epoch = 0
epoch_num = 150
num_sample = 900  # Number of users you want to sample

for epoch in range(epoch_num):  # tuning loop

    if num_sample <= len(known_users_all):
            known_users = known_users_all[torch.randperm(len(known_users_all))[:num_sample]]
    else:
            raise ValueError("Requested sample size exceeds number of known users.")

    known_items = known_items_all

    start_time = time.time()
    train_rmse = train_model(model, train_loader, optimizer, criterion, device, known_users, known_items)
    val_rmse = evaluate_model(model, val_loader, criterion, device)
    if epoch == 0:
        param_bytes = sum(p.numel() for p in model.parameters()) * 4
        comm_cost_epoch = param_bytes
        grad_share_epoch = n_users
        message_count_epoch = 1

    total_comm_cost += comm_cost_epoch
    total_grad_shared += grad_share_epoch
    total_message_count += message_count_epoch
    epoch_time = time.time() - start_time

    # Logging for plotting/export
    comm_cost_log.append(comm_cost_epoch)
    grad_share_log.append(grad_share_epoch)
    message_count_log.append(message_count_epoch)
    epoch_time_log.append(epoch_time)
    rmse_log.append(val_rmse)
    print(f"Epoch {epoch+1}/{epoch_num} - Time: {epoch_time:.2f}s, "
          f"Train RMSE: {val_rmse:.4f}, "
          f"Comm Cost: {comm_cost_epoch}, "
          f"Messages: {message_count_epoch}")

val_rmse = evaluate_model(model, val_loader, criterion, device)
print(f"  => Final Val RMSE: {val_rmse:.4f}")

Epoch 1/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 2/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 3/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 4/150 - Time: 0.24s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 5/150 - Time: 0.27s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 6/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 7/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 8/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 9/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 10/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 11/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 12/150 - Time: 0.26s, Train RMSE: 3.7041, Comm Cost: 340036, Messages: 1
Epoch 13/150 - Time: 0.25s, Train RMSE: 3.7041, Comm Cost: 34

In [13]:
log={}
log["rmse"] = rmse_log
log["epoch time"] = epoch_time_log
log["cost"] = grad_share_log

In [14]:
df = pd.DataFrame.from_dict(log, orient="index")
df.index.name = "epoch"
df.reset_index(inplace=True)
df.to_csv("central_sample.csv", index=False)